# WF2020 Risk-Cap Reference Target Screen V1

This notebook follows the calibrated `wf_2020` gate from `wf2020_reference_floor_gate_calibration_v1`. The goal is not to promote or retain an actor. It tests whether the `SOXLBacktest` Qlib reference floor (`risk_cap_35_15_05`) can be converted into a non-leaking training target for the failed `wf_2020` fold, and whether that target is learnable by the current behavior-cloning warm-start surface.

Governance:

- Canonical baseline: `v19reg_v22_budget_usage_bull_guard_v1`.
- Direct comparison baseline: `wf2020_reference_floor_gate_calibration_v1`.
- Reference baseline: `soxlbacktest_qlib_buysell_reference` / `risk_cap_35_15_05`.
- Role: diagnose / target screen.
- Changed layer: `model_selection`, `training_objective`, `bc_warm_start`.
- Acceptance for the next step: build a non-leaking combined target contract, prove BC learnability, keep `promotion_allowed=false`, and require a file-backed replay before any actor retention.


In [1]:

from __future__ import annotations

import json
import math
import os
import random
import sys
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
while not (ROOT / 'AGENTS.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from src.soxl_quant.compat import patch_gym_compat
patch_gym_compat()

from src.soxl_quant.app.state_objective_bc import (
    StateObjectiveBCSampleWeightConfig,
    StateObjectiveBCTrainingConfig,
    build_state_objective_bc_dataset_from_frame,
    evaluate_bc_actor,
    evaluate_bc_actor_slice_fit,
    fit_state_objective_bc_actor_from_frame,
)
from src.soxl_quant.config import load_profile
from src.soxl_quant.data import build_dataset, flatten_feature_frame, init_qlib
from src.soxl_quant.policy.legacy_soxl import replay_legacy_soxl_policy
from src.soxl_quant.settings import QLIB_DATA_PATH

EXPERIMENT = 'wf2020_risk_cap_reference_target_screen_v1'
CARD = ROOT / 'runtime' / 'model_selection' / 'card' / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)
SOURCE_TARGET_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'fold_local_bull_reentry_participation_target_v1'
SOURCE_TARGET_PATH = SOURCE_TARGET_CARD / 'bull_reentry_three_fold_target_contract.csv'
CALIBRATION_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'wf2020_reference_floor_gate_calibration_v1'
GATE_CONTRACT_PATH = CALIBRATION_CARD / 'training_gate_contract.json'
REFERENCE_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'soxlbacktest_qlib_walkforward_baseline_v1'
REFERENCE_DAILY_PATH = REFERENCE_CARD / 'candidate_daily.csv'
REFERENCE_TRADES_PATH = REFERENCE_CARD / 'candidate_trades.csv'

PROFILE_NAME = 'sparse_release_bc_ppo_screen_v1'
SEED = 42
RISK_CAP_CONFIG = {'max_position': 0.35, 'min_position': 0.0, 'buy_step_initial': 0.15, 'buy_step_normal': 0.05}
TRAIN_CONFIG = StateObjectiveBCTrainingConfig(steps=200, lr=0.001, batch_size=128, seed=SEED)
SAMPLE_WEIGHT_CONFIG = StateObjectiveBCSampleWeightConfig(
    rebound_multiplier=4.0,
    downtrend_multiplier=2.0,
    release_gate_multiplier=10.0,
    weight_cap=30.0,
)
GLOBAL_MIN_ACCURACY = 0.80
FOLD_MIN_ACCURACY = 0.64
BC_SLICE_MIN_ACCURACY = 0.60
RISKCAP_BUY_MIN_ACCURACY = 0.70
REQUIRED_SLICES = {
    'risk_cap_reference_active_state': 20,
    'risk_cap_reference_buy_state': 10,
    'release_gate_state': 20,
    'is_downtrend_state': 100,
    'drawdown_control_state': 20,
}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        try:
            return str(value.relative_to(ROOT))
        except ValueError:
            return str(value)
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + '\n', encoding='utf-8')


def selected_target_position(frame: pd.DataFrame) -> np.ndarray:
    return np.where(
        frame['is_downtrend_state'].fillna(False).astype(bool),
        pd.to_numeric(frame['risk_target_position'], errors='coerce').fillna(0.0),
        np.where(
            frame['is_rebound_cluster'].fillna(False).astype(bool),
            pd.to_numeric(frame['rebound_target_position'], errors='coerce').fillna(0.0),
            pd.to_numeric(frame['teacher_position'], errors='coerce').fillna(0.0),
        ),
    )


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

print('root', ROOT)
print('card', CARD)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


root /Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix
card /Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/runtime/model_selection/card/wf2020_risk_cap_reference_target_screen_v1


## Inputs

The screen uses only fold-local training dates for the candidate target contracts. The `risk_cap_35_15_05` policy is replayed on `2010-03-11` to `2019-12-31` to create the `wf_2020_train` target overlay. The 2020 risk-cap trade path is used only as a reference feature readout and calibrated floor; it is not inserted into `wf_2020_train` labels.


In [2]:

init_qlib(QLIB_DATA_PATH)
profile = load_profile(PROFILE_NAME)
dataset = build_dataset(profile_name=profile.name, feature_packs=tuple(profile.feature.packs))
feature_frame = flatten_feature_frame(dataset.handler.feature_frame).droplevel('instrument').copy()

base_targets = pd.read_csv(SOURCE_TARGET_PATH)
base_targets['datetime'] = pd.to_datetime(base_targets['datetime'])
for col in ('is_rebound_cluster', 'is_downtrend_state', 'release_gate_state', 'drawdown_control_state'):
    if col in base_targets.columns:
        base_targets[col] = base_targets[col].fillna(False).astype(bool)

training_gate_contract = json.loads(GATE_CONTRACT_PATH.read_text(encoding='utf-8'))
reference_daily = pd.read_csv(REFERENCE_DAILY_PATH, parse_dates=['date'])
reference_trades = pd.read_csv(REFERENCE_TRADES_PATH, parse_dates=['date'])

riskcap_train = replay_legacy_soxl_policy(
    start='2010-03-11',
    end='2019-12-31',
    config_overrides=RISK_CAP_CONFIG,
)
riskcap_train_daily = riskcap_train.daily.copy()
riskcap_train_daily['datetime'] = pd.to_datetime(riskcap_train_daily['date'])
riskcap_train_features = riskcap_train_daily[[
    'datetime', 'position', 'action', 'panic_score', 'HV_20', 'VOL_COEF'
]].rename(columns={
    'position': 'riskcap_position',
    'action': 'riskcap_action',
    'panic_score': 'riskcap_panic_score',
    'HV_20': 'riskcap_HV_20',
    'VOL_COEF': 'riskcap_VOL_COEF',
})

coverage = pd.DataFrame([
    {
        'source': 'base_bull_reentry_targets',
        'rows': int(len(base_targets)),
        'min_date': base_targets['datetime'].min().date().isoformat(),
        'max_date': base_targets['datetime'].max().date().isoformat(),
    },
    {
        'source': 'riskcap_train_replay',
        'rows': int(len(riskcap_train_daily)),
        'min_date': riskcap_train_features['datetime'].min().date().isoformat(),
        'max_date': riskcap_train_features['datetime'].max().date().isoformat(),
    },
    {
        'source': 'feature_frame_rebuilt_from_qlib',
        'rows': int(len(feature_frame)),
        'min_date': pd.Timestamp(feature_frame.index.min()).date().isoformat(),
        'max_date': pd.Timestamp(feature_frame.index.max()).date().isoformat(),
    },
])
coverage.to_csv(CARD / 'input_coverage.csv', index=False)
display(coverage)

riskcap_train_metrics = dict(riskcap_train.metrics)
riskcap_train_metrics


[86770:MainThread](2026-06-14 08:59:48,652) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[86770:MainThread](2026-06-14 08:59:49,075) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[86770:MainThread](2026-06-14 08:59:49,075) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/data/qlib/soxl')}


[86770:MainThread](2026-06-14 08:59:50,464) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[86770:MainThread](2026-06-14 08:59:50,465) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[86770:MainThread](2026-06-14 08:59:50,465) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


,source,rows,min_date,max_date
0,base_bull_reentry_targets,3731,2010-03-11,2022-12-30
1,riskcap_train_replay,2470,2010-03-11,2019-12-31
2,feature_frame_rebuilt_from_qlib,3835,2011-03-11,2026-06-10


{'requested_start': '2010-03-11',
 'start': '2010-03-11',
 'end': '2019-12-31',
 'rows': 2470,
 'initial_capital': 100000.0,
 'final_capital': 172299.4185143113,
 'soxl_buyhold_return_pct': 2673.8581694596623,
 'rulebase_return_pct': 72.29941851431133,
 'rulebase_max_drawdown_pct': 17.119681626879,
 'average_position_pct': 6.286012145748988,
 'end_position_pct': 0.7000000000000001,
 'trade_count': 56}

## Buy/Sell Feature Contrast

This section writes a compact buy/sell feature comparison. It compares:

- 2020 reference risk-cap BUY/SELL events.
- Non-leaking 2010-2018/2019 risk-cap training BUY/SELL events.
- Existing bull/reentry target release/downtrend states.

The point is to expose what the model would need to learn before another training run, not to certify promotion.


In [3]:

FEATURE_COLS = [
    'return', 'cum_drop_20', 'cum_drop_30', 'drawdown_21', 'drawdown_63',
    'realized_vol_21', 'vol_coef_21', 'vix_zscore', 'vix_percentile_252',
    'relative_volume_zscore', 'rsi_6', 'rsi_14', 'mom_5', 'mom_10', 'mom_21',
    'regime_risk', 'downside_ratio_21', 'loss_streak_21', 'gain_streak_10',
]
available_features = [col for col in FEATURE_COLS if col in feature_frame.columns]
feature_by_date = feature_frame[available_features].copy()
feature_by_date.index = pd.to_datetime(feature_by_date.index).normalize()

riskcap_2020_daily = reference_daily[
    reference_daily['candidate'].eq('risk_cap_35_15_05')
    & reference_daily['window'].eq('wf_2020')
].copy()
riskcap_2020_daily['datetime'] = pd.to_datetime(riskcap_2020_daily['date']).dt.normalize()
riskcap_2020_trades = reference_trades[
    reference_trades['candidate'].eq('risk_cap_35_15_05')
    & reference_trades['window'].eq('wf_2020')
].copy()
riskcap_2020_trades['datetime'] = pd.to_datetime(riskcap_2020_trades['date']).dt.normalize()

riskcap_train_trades = riskcap_train.trades.copy()
riskcap_train_trades['datetime'] = pd.to_datetime(riskcap_train_trades['date']).dt.normalize()

train_target = base_targets[base_targets['window'].eq('wf_2020_train')].copy()
train_target['datetime'] = pd.to_datetime(train_target['datetime']).dt.normalize()
train_target['selected_target_position'] = selected_target_position(train_target)

def event_feature_rows(events: pd.DataFrame, label: str, event_col: str = 'type') -> pd.DataFrame:
    if events.empty:
        return pd.DataFrame()
    rows = events.copy()
    rows['group'] = label
    rows['event_type'] = rows[event_col].astype(str).str.upper()
    rows['datetime'] = pd.to_datetime(rows['datetime']).dt.normalize()
    merged = rows.merge(feature_by_date.reset_index().rename(columns={'datetime': 'datetime'}), on='datetime', how='left')
    keep_cols = ['group', 'event_type', 'datetime', *[c for c in ['price', 'position', 'score', 'gain'] if c in merged.columns], *available_features]
    return merged[keep_cols]

feature_events = pd.concat([
    event_feature_rows(riskcap_2020_trades, 'riskcap_wf2020_reference_events'),
    event_feature_rows(riskcap_train_trades, 'riskcap_wf2020_train_events'),
], ignore_index=True)
feature_events.to_csv(CARD / 'buy_sell_feature_events.csv', index=False)

group_frames = []
if not feature_events.empty:
    group_frames.append(feature_events)
release_rows = train_target[train_target['release_gate_state'].astype(bool)].copy()
release_rows['group'] = 'current_train_release_gate_rows'
release_rows['event_type'] = 'RELEASE_TARGET'
downtrend_rows = train_target[train_target['is_downtrend_state'].astype(bool)].copy()
downtrend_rows['group'] = 'current_train_downtrend_rows'
downtrend_rows['event_type'] = 'DOWNTREND_TARGET'
for rows in (release_rows, downtrend_rows):
    rows = rows.merge(feature_by_date.reset_index(), on='datetime', how='left')
    rows['position'] = rows['selected_target_position']
    group_frames.append(rows[['group', 'event_type', 'datetime', 'position', *available_features]])
all_feature_rows = pd.concat(group_frames, ignore_index=True, sort=False)
summary_rows = []
for (group, event_type), subset in all_feature_rows.groupby(['group', 'event_type']):
    row = {'group': group, 'event_type': event_type, 'rows': int(len(subset))}
    if 'position' in subset.columns:
        row['mean_position_or_target'] = float(pd.to_numeric(subset['position'], errors='coerce').mean())
    for col in available_features:
        row[f'{col}_mean'] = float(pd.to_numeric(subset[col], errors='coerce').mean())
    summary_rows.append(row)
feature_summary = pd.DataFrame(summary_rows).sort_values(['group', 'event_type']).reset_index(drop=True)
feature_summary.to_csv(CARD / 'buy_sell_feature_comparison.csv', index=False)
display(feature_summary[['group', 'event_type', 'rows', 'mean_position_or_target', *[f'{c}_mean' for c in available_features[:6]]]])


,group,event_type,rows,mean_position_or_target,return_mean,cum_drop_20_mean,cum_drop_30_mean,drawdown_21_mean,drawdown_63_mean,realized_vol_21_mean
0,current_train_downtrend_rows,DOWNTREND_TARGET,781,0.264986,-0.003128,-0.145243,-0.179200,-0.148980,-0.259499,0.805109
1,current_train_release_gate_rows,RELEASE_TARGET,42,0.657243,-0.060637,-0.196704,-0.226249,-0.199859,-0.251807,0.784475
2,riskcap_wf2020_reference_events,BUY,4,0.171300,-0.271145,-0.600252,-0.600252,-0.600252,-0.600252,1.799386
3,riskcap_wf2020_train_events,BUY,31,0.107094,-0.096716,-0.270500,-0.307295,-0.274016,-0.334935,0.802122
4,riskcap_wf2020_train_events,SELL,25,0.075520,0.037842,-0.016353,-0.032401,-0.017140,-0.086539,0.759614


## Candidate Target Contracts

The screen compares four `wf_2020_train` target semantics while leaving `wf_2022_train` and `wf_2023_2024_train` unchanged:

- `current_bull_reentry_contract`: existing target, for baseline fit only.
- `riskcap_full_replacement`: replace all `wf_2020_train` target positions with the non-leaking `risk_cap_35_15_05` replay path. This is a lower-bound target and likely too conservative for return alignment.
- `riskcap_active10_replacement`: replace only dates where the risk-cap replay is at least 10% exposed or explicitly buys/sells.
- `riskcap_crash_cap`: cap current targets on crash/high-volatility states but preserve more ordinary bull/reentry exposure.


In [4]:

def make_contract_variant(name: str) -> pd.DataFrame:
    out = base_targets.copy()
    out['target_variant'] = name
    for col in ('risk_cap_reference_active_state', 'risk_cap_reference_buy_state', 'risk_cap_reference_sell_state'):
        out[col] = False

    wf_mask = out['window'].eq('wf_2020_train')
    merged = out.loc[wf_mask].merge(riskcap_train_features, on='datetime', how='left')
    active = merged['riskcap_position'].fillna(0.0).ge(0.10) | merged['riskcap_action'].isin(['BUY', 'SELL'])
    buy = merged['riskcap_action'].eq('BUY')
    sell = merged['riskcap_action'].eq('SELL')
    if name == 'current_bull_reentry_contract':
        changed = pd.Series(False, index=merged.index)
    elif name == 'riskcap_full_replacement':
        changed = merged['riskcap_position'].notna()
    elif name == 'riskcap_active10_replacement':
        changed = active
    elif name == 'riskcap_crash_cap':
        changed = (
            merged['riskcap_panic_score'].le(-50.0)
            | merged['riskcap_HV_20'].ge(0.80)
            | merged['riskcap_position'].fillna(0.0).ge(0.10)
        )
    else:
        raise ValueError(f'unknown contract variant: {name}')

    if name in {'riskcap_full_replacement', 'riskcap_active10_replacement'}:
        for col in ('teacher_position', 'rebound_target_position', 'risk_target_position', 'risk_target_position_original'):
            merged.loc[changed, col] = merged.loc[changed, 'riskcap_position'].fillna(0.0).clip(0.0, 0.35)
    elif name == 'riskcap_crash_cap':
        cap = np.maximum(
            merged['riskcap_position'].fillna(0.0).clip(0.0, 0.35),
            np.where(buy, 0.15, 0.0),
        )
        for col in ('teacher_position', 'rebound_target_position', 'risk_target_position', 'risk_target_position_original'):
            merged.loc[changed, col] = np.minimum(
                pd.to_numeric(merged.loc[changed, col], errors='coerce').fillna(0.0),
                cap[changed],
            )

    merged.loc[changed, 'objective_weight'] = np.maximum(
        pd.to_numeric(merged.loc[changed, 'objective_weight'], errors='coerce').fillna(1.0),
        np.where(buy[changed], 12.0, np.where(sell[changed], 10.0, 8.0)),
    )
    merged.loc[changed, 'risk_cap_reference_active_state'] = active[changed].to_numpy()
    merged.loc[changed, 'risk_cap_reference_buy_state'] = buy[changed].to_numpy()
    merged.loc[changed, 'risk_cap_reference_sell_state'] = sell[changed].to_numpy()
    if name != 'current_bull_reentry_contract':
        merged.loc[changed & active, 'is_downtrend_state'] = True
        merged.loc[changed & active, 'drawdown_control_state'] = True
        merged.loc[changed & buy, 'release_gate_state'] = True
        merged.loc[changed & buy, 'is_rebound_cluster'] = False
        merged.loc[changed, 'source_components'] = merged.loc[changed, 'source_components'].astype(str) + ',risk_cap_reference_floor'
        merged.loc[changed, 'source_contract_id'] = EXPERIMENT
        merged.loc[changed, 'target_semantics_revision'] = name

    columns = list(out.columns)
    out.loc[wf_mask, columns] = merged[columns].to_numpy()
    return out

variant_names = [
    'current_bull_reentry_contract',
    'riskcap_full_replacement',
    'riskcap_active10_replacement',
    'riskcap_crash_cap',
]
contracts = {name: make_contract_variant(name) for name in variant_names}

base_wf = base_targets[base_targets['window'].eq('wf_2020_train')].copy()
base_target_position = selected_target_position(base_wf)

distribution_rows = []
for name, contract in contracts.items():
    wf = contract[contract['window'].eq('wf_2020_train')].copy()
    target_position = selected_target_position(wf)
    distribution_rows.append({
        'variant': name,
        'wf2020_train_rows': int(len(wf)),
        'avg_target_position': float(np.mean(target_position)),
        'median_target_position': float(np.median(target_position)),
        'target_ge_35_rows': int((target_position >= 0.35).sum()),
        'target_ge_70_rows': int((target_position >= 0.70).sum()),
        'riskcap_active_rows': int(wf['risk_cap_reference_active_state'].fillna(False).astype(bool).sum()),
        'riskcap_buy_rows': int(wf['risk_cap_reference_buy_state'].fillna(False).astype(bool).sum()),
        'changed_rows_vs_current': int((np.abs(target_position - base_target_position) > 1e-9).sum()),
    })

target_distribution = pd.DataFrame(distribution_rows)
target_distribution.to_csv(CARD / 'candidate_target_distribution.csv', index=False)
for name in variant_names:
    contracts[name].to_csv(CARD / f'{name}_combined_target_contract.csv', index=False)
contract_manifest = pd.DataFrame([
    {'variant': name, 'path': f'runtime/model_selection/card/{EXPERIMENT}/{name}_combined_target_contract.csv'}
    for name in variant_names
])
contract_manifest.to_csv(CARD / 'candidate_contract_manifest.csv', index=False)
display(target_distribution)


,variant,wf2020_train_rows,avg_target_position,median_target_position,target_ge_35_rows,target_ge_70_rows,riskcap_active_rows,riskcap_buy_rows,changed_rows_vs_current
0,current_bull_reentry_contract,2470,0.330183,0.2500,966,140,0,0,0
1,riskcap_full_replacement,2470,0.062280,0.0343,49,0,424,31,2470
2,riskcap_active10_replacement,2470,0.302384,0.2500,898,84,424,31,424
3,riskcap_crash_cap,2470,0.263596,0.2500,779,36,413,31,680


## BC Learnability Screen

The screen trains short behavior-cloning warm starts for each target variant and measures whether labels can be fit on all folds plus risk-cap-specific slices. This is not a performance backtest and cannot justify promotion. It only decides whether the target contract is coherent enough to hand off to a file-backed replay.


In [5]:

def slice_gate_from_metrics(slice_metrics: pd.DataFrame) -> tuple[bool, dict[str, Any]]:
    indexed = slice_metrics.set_index('slice') if not slice_metrics.empty else pd.DataFrame()
    details: dict[str, Any] = {}
    gate = True
    for slice_name, min_samples in REQUIRED_SLICES.items():
        if slice_name not in indexed.index:
            row = {'samples': 0, 'accuracy': float('nan'), 'gate': False}
        else:
            item = indexed.loc[slice_name]
            samples = int(item.get('samples', 0))
            accuracy = float(item.get('accuracy', float('nan')))
            threshold = RISKCAP_BUY_MIN_ACCURACY if slice_name == 'risk_cap_reference_buy_state' else BC_SLICE_MIN_ACCURACY
            row = {
                'samples': samples,
                'accuracy': accuracy,
                'min_samples': int(min_samples),
                'min_accuracy': float(threshold),
                'gate': bool(samples >= int(min_samples) and math.isfinite(accuracy) and accuracy >= float(threshold)),
            }
        row.setdefault('min_samples', int(min_samples))
        row.setdefault('min_accuracy', RISKCAP_BUY_MIN_ACCURACY if slice_name == 'risk_cap_reference_buy_state' else BC_SLICE_MIN_ACCURACY)
        details[slice_name] = row
        gate = gate and bool(row['gate'])
    return bool(gate), details

screen_rows: list[dict[str, Any]] = []
fold_rows: list[dict[str, Any]] = []
slice_rows: list[dict[str, Any]] = []
action_rows: list[dict[str, Any]] = []

for variant_name, contract in contracts.items():
    per_fold_accuracy: list[float] = []
    per_fold_samples: list[int] = []
    wf2020_slice_gate = False
    wf2020_slice_details: dict[str, Any] = {}

    for fold, fold_targets in contract.groupby('fold'):
        fold = str(fold)
        set_seed(SEED)
        bc_dataset = build_state_objective_bc_dataset_from_frame(
            feature_frame,
            targets=fold_targets,
            profile=profile,
            sample_size=0,
        )
        fit = fit_state_objective_bc_actor_from_frame(
            feature_frame,
            targets=fold_targets,
            profile=profile,
            train_config=TRAIN_CONFIG,
            sample_weight_config=SAMPLE_WEIGHT_CONFIG,
            sample_size=0,
        )
        fold_eval = evaluate_bc_actor(fit.actor, bc_dataset.obs, bc_dataset.actions)
        fold_accuracy = float(fold_eval['accuracy'])
        fold_samples = int(fold_eval['samples'])
        per_fold_accuracy.append(fold_accuracy)
        per_fold_samples.append(fold_samples)
        fold_rows.append({
            'variant': variant_name,
            'fold': fold,
            'samples': fold_samples,
            'accuracy': fold_accuracy,
            'loss': float(fold_eval['loss']),
            'fold_gate': bool(fold_accuracy >= FOLD_MIN_ACCURACY),
            'before_accuracy': float(fit.metrics['before_accuracy']),
            'after_accuracy': float(fit.metrics['after_accuracy']),
        })

        metrics = evaluate_bc_actor_slice_fit(
            fit.actor,
            bc_dataset.obs,
            bc_dataset.actions,
            bc_dataset.metadata,
            slice_cols=tuple(REQUIRED_SLICES),
        )
        metrics.insert(0, 'fold', fold)
        metrics.insert(0, 'variant', variant_name)
        slice_rows.extend(metrics.to_dict('records'))
        if fold == 'wf_2020_train':
            wf2020_slice_gate, wf2020_slice_details = slice_gate_from_metrics(metrics)

        dist = fit.action_distribution.copy()
        dist.insert(0, 'fold', fold)
        dist.insert(0, 'variant', variant_name)
        action_rows.extend(dist.to_dict('records'))

    weighted_accuracy = float(np.average(per_fold_accuracy, weights=per_fold_samples))
    min_fold_accuracy = float(min(per_fold_accuracy))
    global_gate = bool(weighted_accuracy >= GLOBAL_MIN_ACCURACY)
    fold_gate = bool(min_fold_accuracy >= FOLD_MIN_ACCURACY)
    gate_loss = 0.0
    gate_loss += max(0.0, GLOBAL_MIN_ACCURACY - weighted_accuracy)
    gate_loss += max(0.0, FOLD_MIN_ACCURACY - min_fold_accuracy)
    for detail in wf2020_slice_details.values():
        accuracy = float(detail.get('accuracy', float('nan')))
        min_accuracy = float(detail.get('min_accuracy', BC_SLICE_MIN_ACCURACY))
        samples = int(detail.get('samples', 0))
        min_samples = int(detail.get('min_samples', 0))
        if not math.isfinite(accuracy):
            gate_loss += 1.0
        else:
            gate_loss += max(0.0, min_accuracy - accuracy)
        gate_loss += max(0.0, min_samples - samples) / max(min_samples, 1)

    screen_rows.append({
        'variant': variant_name,
        'weighted_accuracy': weighted_accuracy,
        'min_fold_accuracy': min_fold_accuracy,
        'global_accuracy_gate': global_gate,
        'fold_accuracy_gate': fold_gate,
        'wf2020_slice_gate': bool(wf2020_slice_gate),
        'bc_screen_gate': bool(global_gate and fold_gate and wf2020_slice_gate),
        'gate_loss': float(gate_loss),
        'train_steps': int(TRAIN_CONFIG.steps),
        'sample_weight_config': asdict(SAMPLE_WEIGHT_CONFIG),
        'wf2020_slice_details': wf2020_slice_details,
    })

bc_screen = pd.DataFrame(screen_rows).sort_values(['bc_screen_gate', 'gate_loss'], ascending=[False, True]).reset_index(drop=True)
fold_metrics = pd.DataFrame(fold_rows)
slice_metrics = pd.DataFrame(slice_rows)
action_distribution = pd.DataFrame(action_rows)

bc_screen.to_csv(CARD / 'bc_target_screen.csv', index=False)
fold_metrics.to_csv(CARD / 'bc_fold_metrics.csv', index=False)
slice_metrics.to_csv(CARD / 'bc_slice_metrics.csv', index=False)
action_distribution.to_csv(CARD / 'bc_action_distribution.csv', index=False)
write_json(CARD / 'bc_screen_summary.json', {
    'experiment': EXPERIMENT,
    'profile': PROFILE_NAME,
    'train_config': asdict(TRAIN_CONFIG),
    'sample_weight_config': asdict(SAMPLE_WEIGHT_CONFIG),
    'global_min_accuracy': GLOBAL_MIN_ACCURACY,
    'fold_min_accuracy': FOLD_MIN_ACCURACY,
    'required_slices': REQUIRED_SLICES,
    'riskcap_buy_min_accuracy': RISKCAP_BUY_MIN_ACCURACY,
    'screen_rows': screen_rows,
})

display(bc_screen[['variant', 'weighted_accuracy', 'min_fold_accuracy', 'global_accuracy_gate', 'fold_accuracy_gate', 'wf2020_slice_gate', 'bc_screen_gate', 'gate_loss']])


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


...

/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


,variant,weighted_accuracy,min_fold_accuracy,global_accuracy_gate,fold_accuracy_gate,wf2020_slice_gate,bc_screen_gate,gate_loss
0,riskcap_full_replacement,0.824166,0.646825,True,True,True,True,0.000000
1,riskcap_crash_cap,0.573435,0.510895,False,False,False,False,0.720183
2,riskcap_active10_replacement,0.643359,0.621697,False,False,False,False,0.820937
3,current_bull_reentry_contract,0.676126,0.646825,False,True,False,False,4.123874


## Decision

The selected screen row is a lower-bound target handoff. It is allowed to feed the next file-backed replay, but it is not a retained model and not a proof of return alignment. Actor retention still requires the calibrated walk-forward gate plus the current-best/SOXLBacktest YTD return floors.


In [6]:

passed = bc_screen[bc_screen['bc_screen_gate'].astype(bool)].copy()
selected = (passed if not passed.empty else bc_screen).sort_values(['gate_loss']).iloc[0].to_dict()
selected_variant = str(selected['variant'])
selected_distribution = target_distribution[target_distribution['variant'].eq(selected_variant)].iloc[0].to_dict()

return_exposure_warning = bool(float(selected_distribution['avg_target_position']) < 0.20)
if bool(selected['bc_screen_gate']):
    decision = 'wf2020_riskcap_reference_target_bc_lower_bound_ready_no_promotion'
    next_mechanism = 'run_file_backed_wf2020_riskcap_full_bc_replay_before_any_retention_then_design_return_preserving_hybrid_target_if_ytd_gate_fails'
else:
    decision = 'wf2020_riskcap_reference_target_screen_not_learnable_no_training'
    next_mechanism = 'revise_riskcap_active_state_target_before_checkpoint_replay'

selection = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': decision,
    'selected_variant': selected_variant,
    'selected_contract_path': f'runtime/model_selection/card/{EXPERIMENT}/{selected_variant}_combined_target_contract.csv',
    'selected_bc_screen': selected,
    'selected_target_distribution': selected_distribution,
    'target_contract_gate': bool(selected['bc_screen_gate']),
    'bc_readiness_gate': bool(selected['bc_screen_gate']),
    'bc_screen_allowed_next': bool(selected['bc_screen_gate']),
    'training_allowed_from_this_card': False,
    'promotion_allowed': False,
    'actor_retention_allowed': False,
    'promotion_blocker': 'Target screen and BC learnability only. No actor replay, YTD return gate, fixed-split gate, or full walk-forward replay passed in this card.',
    'return_exposure_warning': return_exposure_warning,
    'return_exposure_note': 'Selected risk-cap full replacement is intentionally conservative and may fail current-best/YTD return alignment; next replay must prove or reject it before retention.',
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': 'wf2020_reference_floor_gate_calibration_v1',
    'reference_baseline': 'soxlbacktest_qlib_buysell_reference:risk_cap_35_15_05',
    'source_decisions': {
        'wf2020_reference_floor_gate_calibration_v1': 'wf2020_reference_floor_gate_calibrated_no_promotion',
        'fold_local_bull_reentry_participation_target_v1': 'target_revision_ready_for_three_fold_bc_checkpoint_metric_replay',
    },
    'role': 'diagnose',
    'changed_layer': ['model_selection', 'training_objective', 'bc_warm_start'],
    'target_relative_return_by_fold_pp': training_gate_contract['target_relative_return_by_fold_pp'],
    'target_max_dd_pct': training_gate_contract['max_walkforward_drawdown_pct'],
    'required_best_return_pct': training_gate_contract['required_ytd_return_pct'],
    'required_soxlbacktest_return_pct': training_gate_contract['required_soxlbacktest_reference_return_pct'],
    'wf2020_reference_floor_return_pct': training_gate_contract['wf2020_reference_floor_return_pct'],
    'wf2020_reference_floor_max_dd_pct': training_gate_contract['wf2020_reference_floor_max_dd_pct'],
    'next_mechanism': next_mechanism,
    'sidecars': [
        'input_coverage.csv',
        'buy_sell_feature_events.csv',
        'buy_sell_feature_comparison.csv',
        'candidate_target_distribution.csv',
        'candidate_contract_manifest.csv',
        'bc_target_screen.csv',
        'bc_fold_metrics.csv',
        'bc_slice_metrics.csv',
        'bc_action_distribution.csv',
        'bc_screen_summary.json',
        'selection.json',
        'summary.json',
        'conclusion.md',
    ],
}
summary = {
    **selection,
    'riskcap_train_metrics': riskcap_train_metrics,
    'screen_rows': bc_screen.to_dict('records'),
    'target_distribution_rows': target_distribution.to_dict('records'),
    'next_training_allowed_after_target_revision': bool(selected['bc_screen_gate']),
}
write_json(CARD / 'selection.json', selection)
write_json(CARD / 'summary.json', summary)

conclusion_lines = [
    f"# {EXPERIMENT}",
    '',
    f"Decision: `{decision}`.",
    '',
    f"Selected variant: `{selected_variant}`.",
    f"BC weighted accuracy `{float(selected['weighted_accuracy']):.3f}`, min fold accuracy `{float(selected['min_fold_accuracy']):.3f}`, gate `{bool(selected['bc_screen_gate'])}`.",
    f"Selected wf2020 avg target position `{float(selected_distribution['avg_target_position']):.3f}` with `{int(selected_distribution['target_ge_35_rows'])}` rows at target >=35% and `{int(selected_distribution['target_ge_70_rows'])}` rows at target >=70%.",
    '',
    'Key finding: the risk-cap full replacement is learnable and provides a non-leaking lower-bound target, but it is intentionally conservative and still needs file-backed replay before any retention decision. The return-preserving active/crash variants did not pass the BC screen, so they should not be used for PPO/checkpoint replay yet.',
    '',
    'Promotion note: `promotion_allowed=false` and `actor_retention_allowed=false`; the next replay must still satisfy the calibrated wf2020 floor, wf2022/wf2023_2024 +30pp relative gates, MaxDD <20%, and YTD/common-window return floors.',
    '',
    f"Next action: `{next_mechanism}`.",
    '',
    'Artifacts: `candidate_target_distribution.csv`, `buy_sell_feature_comparison.csv`, `bc_target_screen.csv`, `bc_fold_metrics.csv`, `bc_slice_metrics.csv`, `selection.json`, `summary.json`.',
]
conclusion = '\n'.join(conclusion_lines) + '\n'
(CARD / 'conclusion.md').write_text(conclusion, encoding='utf-8')
display(Markdown(conclusion))


# wf2020_risk_cap_reference_target_screen_v1

Decision: `wf2020_riskcap_reference_target_bc_lower_bound_ready_no_promotion`.

Selected variant: `riskcap_full_replacement`.
BC weighted accuracy `0.824`, min fold accuracy `0.647`, gate `True`.
Selected wf2020 avg target position `0.062` with `49` rows at target >=35% and `0` rows at target >=70%.

Key finding: the risk-cap full replacement is learnable and provides a non-leaking lower-bound target, but it is intentionally conservative and still needs file-backed replay before any retention decision. The return-preserving active/crash variants did not pass the BC screen, so they should not be used for PPO/checkpoint replay yet.

Promotion note: `promotion_allowed=false` and `actor_retention_allowed=false`; the next replay must still satisfy the calibrated wf2020 floor, wf2022/wf2023_2024 +30pp relative gates, MaxDD <20%, and YTD/common-window return floors.

Next action: `run_file_backed_wf2020_riskcap_full_bc_replay_before_any_retention_then_design_return_preserving_hybrid_target_if_ytd_gate_fails`.

Artifacts: `candidate_target_distribution.csv`, `buy_sell_feature_comparison.csv`, `bc_target_screen.csv`, `bc_fold_metrics.csv`, `bc_slice_metrics.csv`, `selection.json`, `summary.json`.
